# 00 · Smoke Test del Modelo NMT Base (Helsinki-NLP/opus-mt-es-en)

**Proyecto:** Traducción de Mensajes de Audio con Clonación de Voz mediante IA  
**Módulo:** Traducción Automática Neuronal (NMT)  
**Modelo:** `Helsinki-NLP/opus-mt-es-en`  

> **AVISO IMPORTANTE:** Este notebook realiza un **SMOKE TEST** de inferencia inicial sobre 5 muestras de prueba de OPUS-100 para validar que el modelo base, el tokenizador y el entorno de ejecución (CPU/GPU) funcionan correctamente. **Las métricas calculadas aquí NO representan la evaluación cuantitativa final del proyecto.**

## 1. Configuración del Entorno y Dependencias

In [1]:
# 1. Instalar dependencias si se ejecuta en Google Colab
%pip install --quiet "transformers>=4.38.0" "datasets>=3.0.0" "sacrebleu" "evaluate" "sentencepiece" "sacremoses" "pyyaml"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.9 MB/s eta 0:00:00


In [2]:
# 2. Directorios y entorno de trabajo
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Setrox1250/Proyecto-IA-Traductor.git"
REPO_DIR = Path("/content/Proyecto-IA-Traductor")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    PROJECT_ROOT = REPO_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Directorio de trabajo: {PROJECT_ROOT}")

Directorio de trabajo: /content/Proyecto-IA-Traductor


In [3]:
# 3. Detectar hardware disponible
import torch

device_name = "CPU"
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_mb = round(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024), 2)
    print(f"GPU detectada: {device_name} ({vram_mb} MB VRAM)")
    print(f"Versión PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
else:
    print("GPU no disponible. Ejecutando en CPU.")
    print("TIP Colab: Para acelerar la inferencia, ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' -> 'T4 GPU'.")

GPU detectada: Tesla T4 (14912.69 MB VRAM)
Versión PyTorch: 2.11.0+cu130 | CUDA: 13.0


## 2. Carga de Muestras del Split TEST de OPUS-100

In [4]:
import unicodedata
import pandas as pd
from datasets import load_dataset

def normalize_text(text: str) -> str:
    """Normalización NFKC y limpieza de espacios redundantes."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    return " ".join(text.split()).strip()

NUM_SAMPLES = 5
test_csv_path = PROJECT_ROOT / "data/processed/translation/test.csv"
samples = []

if test_csv_path.exists():
    print(f"Cargando muestras desde archivo local: {test_csv_path}")
    df_test = pd.read_csv(test_csv_path, nrows=NUM_SAMPLES)
    for _, row in df_test.iterrows():
        samples.append({"source_es": normalize_text(row["text_es"]), "reference_en": normalize_text(row["text_en"])})
else:
    print("Archivo local test.csv no encontrado; cargando split oficial test desde Hugging Face (streaming)...")
    opus_stream = load_dataset("Helsinki-NLP/opus-100", "en-es", split="test", streaming=True)
    for item in opus_stream:
        trans = item.get("translation") or {}
        es = normalize_text(trans.get("es"))
        en = normalize_text(trans.get("en"))
        if es and en:
            samples.append({"source_es": es, "reference_en": en})
            if len(samples) >= NUM_SAMPLES:
                break

print(f"Muestras preparadas: {len(samples)}")

Archivo local test.csv no encontrado; cargando split oficial test desde Hugging Face (streaming)...


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Muestras preparadas: 5


## 3. Inferencia con MarianMT Base (`Helsinki-NLP/opus-mt-es-en`)

In [5]:
import time
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Intenta importar desde módulo local si existe; si no, utiliza definición autónoma en Colab
try:
    from src.translation.model import MarianTranslator
except (ImportError, ModuleNotFoundError):
    class MarianTranslator:
        """Traductor español -> inglés basado en MarianMT (autónomo en Colab)."""
        def __init__(
            self,
            model_name_or_path: str = "Helsinki-NLP/opus-mt-es-en",
            device: str | torch.device | None = None,
            max_length: int = 128,
            num_beams: int = 4,
        ) -> None:
            self.model_name_or_path = model_name_or_path
            self.max_length = max_length
            self.num_beams = num_beams
            if device is None:
                self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            else:
                self.device = torch.device(device)

            print(f"Cargando modelo MarianMT ({self.model_name_or_path}) en {self.device}...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name_or_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name_or_path)
            self.model.to(self.device)
            self.model.eval()

        def translate_batch(
            self,
            texts: list[str],
            batch_size: int = 16,
            max_length: int | None = None,
            num_beams: int | None = None,
        ) -> list[str]:
            if not texts:
                return []
            out_max_length = max_length or self.max_length
            out_num_beams = num_beams or self.num_beams
            results: list[str] = []
            for i in range(0, len(texts), batch_size):
                batch = list(texts[i : i + batch_size])
                inputs = self.tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=out_max_length,
                )
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                with torch.no_grad():
                    generated_tokens = self.model.generate(
                        **inputs,
                        max_length=out_max_length,
                        num_beams=out_num_beams,
                    )
                decoded = self.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
                results.extend([s.strip() for s in decoded])
            return results

# Hiperparámetro inicial provisional: beam search = 4, max_length = 128
translator = MarianTranslator(num_beams=4, max_length=128)
print(f"Modelo inicializado en: {translator.device} (num_beams={translator.num_beams}, max_length={translator.max_length})")

sources = [s["source_es"] for s in samples]
start_time = time.time()
predictions = translator.translate_batch(sources)
total_time = time.time() - start_time

for s, p in zip(samples, predictions):
    s["prediction_base"] = p

print(f"Inferencia completada en {total_time:.4f} s ({total_time / len(samples) * 1000:.2f} ms por oración)")

Cargando modelo MarianMT (Helsinki-NLP/opus-mt-es-en) en cuda...


config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Modelo inicializado en: cuda (num_beams=4, max_length=128)
Inferencia completada en 2.6195 s (523.90 ms por oración)


## 4. Tabla Comparativa de Resultados

In [6]:
df_results = pd.DataFrame(samples)
df_results.columns = ["SOURCE_ES", "REFERENCE_EN", "PREDICTION_BASE"]
pd.set_option("display.max_colwidth", None)
display(df_results)

,SOURCE_ES,REFERENCE_EN,PREDICTION_BASE
0,"Si su país produjo SAO para estos usos, sírvase anotar en la columna 6 del formulario de datos 3 la cantidad correspondiente”.","If your country produced ODS for this purpose, please enter the amount so produced in column 6 on Data Form 3.”","If your country has produced ODS for these uses, please enter the corresponding amount in column 6 of data form 3.”"
1,"# Juvie el gran hombre, ¿quién podría ser sino yo?","♪ juvie the great man, who else could it be but I?","# Juvie the great man, who could it be but me?"
2,El planeta madre se está agotando.,The home planet is running out.,Mother planet's running low.
3,Las chicas no matan a sus madres? .,Oon't girls ever kill their mothers?,Girls don't kill their mothers.
4,"Actividades humanitarias, de recuperación y de desarrollo","Humanitarian, recovery and development activities","Humanitarian, recovery and development activities"


## 5. Cálculo de Métricas Smoke Test (SacreBLEU y chrF)

In [7]:
import json
import evaluate

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

preds = [s["prediction_base"] for s in samples]
refs = [[s["reference_en"]] for s in samples]

bleu_score = bleu_metric.compute(predictions=preds, references=refs)["score"]
chrf_score = chrf_metric.compute(predictions=preds, references=refs)["score"]

print("=" * 60)
print("SMOKE TEST - ESTAS MÉTRICAS NO REPRESENTAN LA EVALUACIÓN FINAL DEL MODELO")
print(f"Muestras evaluadas: {len(samples)}")
print(f"SacreBLEU: {bleu_score:.2f}")
print(f"chrF: {chrf_score:.2f}")
print("=" * 60)

# Guardar resultados en outputs/smoke_tests/
output_file = PROJECT_ROOT / "outputs/smoke_tests/nmt_base_results.json"
output_file.parent.mkdir(parents=True, exist_ok=True)

report = {
    "model_name": "Helsinki-NLP/opus-mt-es-en",
    "checkpoint": "base",
    "device": str(translator.device),
    "num_samples": len(samples),
    "inference_duration_sec": round(total_time, 4),
    "avg_latency_ms": round((total_time / len(samples)) * 1000, 2),
    "smoke_metrics": {
        "sacrebleu": round(bleu_score, 2),
        "chrf": round(chrf_score, 2),
        "disclaimer": "SMOKE TEST - ESTAS MÉTRICAS NO REPRESENTAN LA EVALUACIÓN FINAL DEL MODELO"
    },
    "samples": samples
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Reporte guardado en: {output_file}")

SMOKE TEST - ESTAS MÉTRICAS NO REPRESENTAN LA EVALUACIÓN FINAL DEL MODELO
Muestras evaluadas: 5
SacreBLEU: 36.33
chrF: 68.82
Reporte guardado en: /content/Proyecto-IA-Traductor/outputs/smoke_tests/nmt_base_results.json
